In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Libraries imported successfully!")

In [ ]:
# =========================================
# COLUMN NAMES
# =========================================

column_names = [
    'Class',
    'Age',
    'Sex',
    'Steroid',
    'Antivirals',
    'Fatigue',
    'Malaise',
    'Anorexia',
    'LiverBig',
    'LiverFirm',
    'SpleenPalpable',
    'Spiders',
    'Ascites',
    'Varices',
    'Bilirubin',
    'AlkPhosphate',
    'Sgot',
    'Albumin',
    'Protime',
    'Histology'
]

# =========================================
# LOAD DATASET
# =========================================

df = pd.read_csv(
    "hepatitis.csv",
    names=column_names
)

print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)


In [ ]:
# =========================================
# (q) DATA CLEANING
# Remove NA, ?, Negative values etc.
# =========================================

# Replace ? with NaN
df.replace("?", np.nan, inplace=True)

# Convert columns to numeric
for col in df.columns:

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )

# Check missing values before cleaning
print("MISSING VALUES BEFORE CLEANING")
print(df.isnull().sum())

# Remove rows with missing values
df_cleaned = df.dropna()

# Remove negative values
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns

df_cleaned = df_cleaned[
    (df_cleaned[numeric_cols] >= 0).all(axis=1)
]

print("\nSHAPE AFTER CLEANING")
print(df_cleaned.shape)


In [ ]:
# =========================================
# (r) ERROR CORRECTING
# OUTLIER DETECTION & REMOVAL (IQR)
# =========================================

def remove_outliers_iqr(df, columns):

    for col in columns:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR

        upper = Q3 + 1.5 * IQR

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

    return df

# Continuous columns used for outlier removal
outlier_cols = [
    'Age',
    'Bilirubin',
    'AlkPhosphate',
    'Sgot',
    'Albumin',
    'Protime'
]

df_no_outliers = remove_outliers_iqr(
    df_cleaned.copy(),
    outlier_cols
)

print("SHAPE AFTER OUTLIER REMOVAL")
print(df_no_outliers.shape)


In [ ]:
# =========================================
# (s) DATA TRANSFORMATION
# FEATURE SCALING
# =========================================

# Features and target
# Class: 1 = DIE, 2 = LIVE
X = df_no_outliers.drop('Class', axis=1)

y = df_no_outliers['Class']

# Map: 1 -> 0 (DIE), 2 -> 1 (LIVE)
y = y.map({1: 0, 2: 1})

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Display transformed dataset
df_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

print("\nTRANSFORMED DATASET")
print(df_scaled.head())


In [ ]:
# =========================================
# TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# (t) LOGISTIC REGRESSION MODEL
# =========================================

logreg = LogisticRegression(
    max_iter=3000,
    class_weight='balanced'
)

logreg.fit(X_train, y_train)

y_pred_logreg = logreg.predict(X_test)

logreg_acc = accuracy_score(
    y_test,
    y_pred_logreg
)

# =========================================
# NAIVE BAYES MODEL
# =========================================

nb = GaussianNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

nb_acc = accuracy_score(
    y_test,
    y_pred_nb
)


In [ ]:
# =========================================
# ACCURACY COMPARISON
# =========================================

print("\n================================")
print("MODEL ACCURACY")
print("================================")

print("\nLogistic Regression Accuracy:")
print(logreg_acc)

print("\nNaive Bayes Accuracy:")
print(nb_acc)

# =========================================
# CLASSIFICATION REPORTS
# =========================================

print("\n================================")
print("LOGISTIC REGRESSION REPORT")
print("================================")

print(classification_report(
    y_test,
    y_pred_logreg,
    target_names=['DIE', 'LIVE']
))

print("\n================================")
print("NAIVE BAYES REPORT")
print("================================")

print(classification_report(
    y_test,
    y_pred_nb,
    target_names=['DIE', 'LIVE']
))

In [ ]:
# =========================================
# CONFUSION MATRIX VISUALIZATION
# LOGISTIC REGRESSION
# =========================================

sns.heatmap(
    confusion_matrix(y_test, y_pred_logreg),
    annot=True,
    fmt='d',
    xticklabels=['DIE', 'LIVE'],
    yticklabels=['DIE', 'LIVE']
)

plt.title("Logistic Regression Confusion Matrix")

plt.show()

# =========================================
# CONFUSION MATRIX VISUALIZATION
# NAIVE BAYES
# =========================================

sns.heatmap(
    confusion_matrix(y_test, y_pred_nb),
    annot=True,
    fmt='d',
    xticklabels=['DIE', 'LIVE'],
    yticklabels=['DIE', 'LIVE']
)

plt.title("Naive Bayes Confusion Matrix")

plt.show()

# =========================================
# FINAL COMPARISON
# =========================================

print("\n================================")
print("FINAL COMPARISON")
print("================================")

if logreg_acc > nb_acc:

    print("\nLogistic Regression Performs Better")

elif nb_acc > logreg_acc:

    print("\nNaive Bayes Performs Better")

else:

    print("\nBoth Models have Equal Accuracy")